In [24]:
# Customer Churn Prediction for a Telecom Company

# Phase 8 – Machine Learning Model Training

# Objective
# The objective of this notebook is to train multiple machine learning classification models for customer churn prediction and compare their performance using various evaluation metrics.

# Tasks
# - Load the preprocessed dataset
# - Train multiple classification models
# - Make predictions
# - Evaluate each model
# - Compare model performance
# - Save the best-performing model

In [25]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [26]:
artifact_root = Path("../artifacts/06_model_training").resolve()
figures_dir = artifact_root / "figures"
tables_dir = artifact_root / "tables"
models_dir = artifact_root / "models"

for directory in [figures_dir, tables_dir, models_dir]:
    directory.mkdir(parents=True, exist_ok=True)

In [27]:
X_train = joblib.load("../data/processed/X_train.pkl")
X_test = joblib.load("../data/processed/X_test.pkl")

y_train = joblib.load("../data/processed/y_train.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

X_train_scaled = joblib.load("../data/processed/X_train_scaled.pkl")
X_test_scaled = joblib.load("../data/processed/X_test_scaled.pkl")

In [28]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "Random Forest": RandomForestClassifier(random_state=42),

    "KNN": KNeighborsClassifier(),

    "SVM": SVC(),

    "Naive Bayes": GaussianNB(),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),

    "LightGBM": LGBMClassifier(random_state=42)
}

In [29]:
results = []
trained_models = {}

for name, model in models.items():

    # Models requiring scaled data
    if name in ["Logistic Regression", "KNN", "SVM"]:

        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)

    else:

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

    trained_models[name] = model

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

    print("="*60)

    print(name)

    print("="*60)

    print(classification_report(y_test, y_pred))

Logistic Regression
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

Decision Tree
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409



Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

KNN
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1035
           1       0.99      0.94      0.97       374

    accuracy                           0.98      1409
   macro avg       0.99      0.97      0.98      1409
weighted avg       0.98      0.98      0.98      1409

SVM
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1035
           1       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      140

In [30]:
results_df = pd.DataFrame(

    results,

    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.000000,1.000000,1.00000,1.00000
1,Decision Tree,1.000000,1.000000,1.00000,1.00000
2,Random Forest,1.000000,1.000000,1.00000,1.00000
3,KNN,0.983676,0.994366,0.94385,0.96845
4,SVM,1.000000,1.000000,1.00000,1.00000
5,Naive Bayes,1.000000,1.000000,1.00000,1.00000
6,XGBoost,1.000000,1.000000,1.00000,1.00000
7,LightGBM,1.000000,1.000000,1.00000,1.00000


In [31]:
metrics_plot_df = results_df.sort_values(by="Accuracy", ascending=False)

ax = metrics_plot_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar",
    figsize=(12, 6),
    rot=45,
    width=0.8
)

ax.set_title("Model Performance Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(figures_dir / "model_performance_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

In [32]:
results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.000000,1.000000,1.00000,1.00000
1,Decision Tree,1.000000,1.000000,1.00000,1.00000
2,Random Forest,1.000000,1.000000,1.00000,1.00000
4,SVM,1.000000,1.000000,1.00000,1.00000
6,XGBoost,1.000000,1.000000,1.00000,1.00000
5,Naive Bayes,1.000000,1.000000,1.00000,1.00000
7,LightGBM,1.000000,1.000000,1.00000,1.00000
3,KNN,0.983676,0.994366,0.94385,0.96845


In [33]:
best_model = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

best_model

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.000000,1.000000,1.00000,1.00000
1,Decision Tree,1.000000,1.000000,1.00000,1.00000
2,Random Forest,1.000000,1.000000,1.00000,1.00000
4,SVM,1.000000,1.000000,1.00000,1.00000
6,XGBoost,1.000000,1.000000,1.00000,1.00000
5,Naive Bayes,1.000000,1.000000,1.00000,1.00000
7,LightGBM,1.000000,1.000000,1.00000,1.00000
3,KNN,0.983676,0.994366,0.94385,0.96845


In [34]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

joblib.dump(
    rf,
    models_dir / "random_forest.pkl"
)

['E:\\Z. Code Programming\\VS Code\\Customer Churn Prediction\\artifacts\\06_model_training\\models\\random_forest.pkl']

In [35]:
results_df.to_excel(
    tables_dir / "model_comparison.xlsx",
    index=False
)

In [36]:
results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.000000,1.000000,1.00000,1.00000
1,Decision Tree,1.000000,1.000000,1.00000,1.00000
2,Random Forest,1.000000,1.000000,1.00000,1.00000
3,SVM,1.000000,1.000000,1.00000,1.00000
4,XGBoost,1.000000,1.000000,1.00000,1.00000
5,Naive Bayes,1.000000,1.000000,1.00000,1.00000
6,LightGBM,1.000000,1.000000,1.00000,1.00000
7,KNN,0.983676,0.994366,0.94385,0.96845


In [37]:
best_index = results_df["Accuracy"].idxmax()
best_model_name = results_df.loc[best_index, "Model"]
best_model_artifact = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")

if best_model_name in ["Logistic Regression", "KNN", "SVM"]:
    best_predictions = best_model_artifact.predict(X_test_scaled)
else:
    best_predictions = best_model_artifact.predict(X_test)

joblib.dump(
    best_model_artifact,
    models_dir / "best_model.pkl"
)

cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(
    figures_dir / f"{best_model_name.lower().replace(' ', '_')}_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

Best Model: Logistic Regression


In [38]:
# Observations

# Eight machine learning classification models were trained and evaluated.

# The performance of each model was compared using:

# - Accuracy
# - Precision
# - Recall
# - F1 Score

# The best-performing model was selected and saved for future deployment.